In [0]:
df = spark.read.format('parquet').load('/Volumes/project/bronze/raw_orders/data')
display(df.limit(10))

In [0]:
df.printSchema()

In [0]:
df = df.withColumnRenamed('_rescued_data','rescued_data')
display(df.limit(10))

In [0]:
df = df.drop('rescued_data')
display(df.limit(10))

In [0]:
from pyspark.sql import functions as F

In [0]:
df = df.withColumn('order_date', F.to_timestamp(F.col('order_date'))).withColumn('year', F.year(F.col('order_date'))).withColumn('month', F.month(F.col('order_date')))
display(df.limit(10))

In [0]:
df_agg = df.groupBy('year','month').agg(F.round(F.sum('total_amount'),2).alias('total_amount'))
df_agg.display()

In [0]:
from pyspark.sql.window import Window

In [0]:
class Windows :
    def __init__(self):
        pass
    
    def apply_rank(self, df, partitions, order_by):
        win = Window.partitionBy(partitions).orderBy(order_by)
        return df.withColumn('rank', F.rank().over(win))
    
    def apply_dense_rank(self, df, partitions, order_by):
        win = Window.partitionBy(partitions).orderBy(order_by)
        return df.withColumn('dense_rank', F.dense_rank().over(win))
    
    def apply_row_number(self, df, partitions, order_by):
        win = Window.partitionBy(partitions).orderBy(order_by)
        return df.withColumn('row_num', F.row_number().over(win))

    def apply_lead(self, df, partitions, order_by,lead_by ,offset = 1):
        win = Window.partitionBy(partitions).orderBy(order_by)
        return df.withColumn('lead', F.lead(F.col(lead_by), offset, default = None).over(win))

    def apply_lag(self, df, partitions, order_by,lag_by, offset = 1):
        win = Window.partitionBy(partitions).orderBy(order_by)
        return df.withColumn('lag', F.lag(F.col(lag_by), offset, default = None).over(win))

In [0]:
obj = Windows()

In [0]:
df1 = obj.apply_rank(df_agg, partitions=F.col('year'), order_by = [F.col('year').desc(), F.col('month').asc(), F.col('total_amount').desc()])
df1.display()

In [0]:
df2 = obj.apply_dense_rank(df1, partitions=F.col('year'), order_by = [F.col('year').desc(), F.col('month').asc(), F.col('total_amount').desc()])
df2.display()

In [0]:
df3 = obj.apply_lead(df2, partitions=F.col('year'), order_by = [F.col('year').desc(), F.col('month').asc(), F.col('total_amount').desc()],lead_by='total_amount')
df3.display()

In [0]:
df4 = obj.apply_lag(df3, partitions=F.col('year'), order_by = [F.col('year').desc(), F.col('month').asc(), F.col('total_amount').desc()],lag_by='total_amount')
display(df4)

In [0]:
df5 = df4.withColumn('running_total_monthly_per_Year', F.round(F.sum('total_amount').over(Window.partitionBy(F.col('year')).orderBy(F.col('month')).rowsBetween(Window.unboundedPreceding, Window.currentRow)), 2))
df5.display()

In [0]:
# df.write.format('delta').mode('overwrite').save('/Volumes/project/volumes/silver_orders_volume/data')


In [0]:
df.write.format('delta').mode('overwrite').saveAsTable('project.silver.orders')

In [0]:
# %sql
# CREATE TABLE IF NOT EXISTS project.silver.orders
# AS 
# SELECT * FROM DELTA.`/Volumes/project/volumes/silver_orders_volume/data`